# 5. Exercise: mixing in a quiet and in a sheared ocean

During PIRATA FR-36 (March 2026), turbulence was measured at 24 stations along 10°W. We compare two of them:

- **9.5°S (station 26)**: away from the equator, where currents are weak;
- **0°N (station 7)**: on the equator, where the Equatorial Undercurrent flows below the surface (part 3).

Is turbulence the same at both stations? Where does it come from? How much heat and oxygen does it move? We go step by step:

1. Compare the two water columns.
2. Check that we can trust $\varepsilon$.
3. Find where the turbulence comes from.
4. Compute the fluxes of heat and oxygen, and compare them with published values.
5. Check whether our two stations are representative of the whole section.

Code to write is marked `...`. The answers are in the dropdowns: try first!

## Step 1. Two water columns

We start with the profiles measured by the VMP. Open the file and look at what it contains.

In [ ]:
import numpy as np
import xarray as xr
import gsw
import matplotlib.pyplot as plt

prof = xr.open_dataset("data/fr36_profiles.nc")
prof

::::{note} Questions
**1.1** How many stations are in the file? On which vertical grid are the profiles? Which variables do they contain?

:::{dropdown} Answers
**1.1** 24 stations, on a grid of 2 dbar from 1 to 249 dbar. Each profile holds temperature `temp`, salinity `sal`, oxygen `o2`, chlorophyll `chla` and the dissipation rate `eps` (medians in 2-dbar bins). Each station also has its latitude, longitude, time and depth of the mixed layer `mld`.
:::
::::

We keep our two stations and compute the potential density $\sigma_\theta$ and the stratification $N^2$ (part 2).

In [ ]:
pres = prof.pres.values
stations = {"stat26": "tab:blue", "stat7": "tab:red"}     # 9.5°S, 0°N

col = {}
for s in stations:
    p = prof.sel(station=s)
    SA = gsw.SA_from_SP(p.sal.values, pres, float(p.lon), float(p.lat))
    CT = gsw.CT_from_t(SA, p.temp.values, pres)
    n2, p_mid = gsw.Nsquared(SA, CT, pres, lat=float(p.lat))
    col[s] = {"temp": p.temp.values, "sal": p.sal.values, "o2": p.o2.values,
              "sigma": gsw.sigma0(SA, CT), "N2": np.interp(pres, p_mid, n2),
              "mld": float(p.mld), "lat": float(p.lat)}

In the tropics, the 20 °C isotherm (D20) is a common marker of the thermocline (part 1). Compute D20 at each station: the first pressure where the temperature is below 20 °C.

In [ ]:
for s in stations:
    col[s]["d20"] = ...

In [ ]:
for s in stations:
    col[s]["d20"] = pres[np.argmax(col[s]["temp"] < 20)]

In [ ]:
fig, axs = plt.subplots(1, 5, figsize=(13, 4.5), sharey=True)
panels = [("temp", "T (°C)"), ("sal", "S"), ("sigma", "σθ (kg m⁻³)"), ("N2", "N² (s⁻²)"), ("o2", "O₂ (µmol kg⁻¹)")]
for s, c in stations.items():
    for ax, (key, label) in zip(axs, panels):
        ax.plot(col[s][key], pres, color=c, label=f"{col[s]['lat']:.1f}°N")
        ax.axhline(col[s]["mld"], color=c, ls="--", lw=1)
        ax.axhline(col[s]["d20"], color=c, ls=":", lw=1)
        ax.set_xlabel(label); ax.grid(alpha=0.3)
axs[0].set_ylim(200, 0); axs[0].set_ylabel("pressure (dbar)"); axs[0].legend()
fig.suptitle("Two water columns. Dashed: base of the mixed layer; dotted: D20");

::::{note} Questions
**1.2** Compare the two stations: thickness of the mixed layer, depth of the thermocline (D20), and where $N^2$ is largest.

**1.3** Salinity has a maximum below the mixed layer at both stations. At which depth?

**1.4** Describe the oxygen below the mixed layer: is there a maximum? Where does oxygen decrease fastest (the oxycline)?

:::{dropdown} Answers
**1.2** At 9.5°S, the mixed layer is about 50 m thick and D20 is near 90 dbar. At the equator, the mixed layer is only about 17 m thick and D20 is near 65 dbar. The thermocline is shallower and sharper at the equator: $N^2$ peaks near 45 dbar, against 55–90 dbar at 9.5°S.

**1.3** Near 75 dbar at 9.5°S, and near 45–60 dbar at the equator. At the equator, this salty water is carried by the Equatorial Undercurrent: we will see the current in step 3.

**1.4** At 9.5°S, oxygen first increases below the mixed layer, to a maximum near 70 dbar, then drops quickly between 85 and 100 dbar, close to D20. At the equator, oxygen stays high down to about 55 dbar, then decreases slowly, without a sharp oxycline.
:::
::::